# 🎯 SigLIP Team Assignment Processor - Performance Evaluation

**Focused evaluation of SigLIP processor performance on real football video data**

This notebook evaluates:
- ⚡ Processing performance (training time, inference time, FPS)
- 🎯 Team assignment accuracy and quality
- 📊 Performance metrics and visualizations
- 💡 Optimization recommendations

---

In [ ]:
# 📦 Essential Imports
import sys
import os
import time
import warnings
from pathlib import Path

# Add project root to path
project_root = Path("/workspaces/football_analysis")
sys.path.append(str(project_root))

# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import torch
from PIL import Image

# Football AI components
from football_ai.assignment.team_assignment_processor import SigLIPTeamAssignmentProcessor
from football_ai.detection.object_detection_processor import ObjectDetectionProcessor
from football_ai.domain.data_models import VideoData, FrameData, Detection, BoundingBox, ObjectType

# Configure plotting
plt.style.use('default')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

print("✅ All imports successful!")
print(f"📁 Project root: {project_root}")
print(f"🖥️ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 🎬 Load Real Video Data
def load_video_data(max_frames=50):
    """Load real video data with YOLO detection"""
    
    # Find video file
    input_dir = project_root / "input_videos"
    video_files = [f for f in input_dir.glob("*.mp4") if f.name != "put_input_video_here.txt"]
    if not video_files:
        raise FileNotFoundError("❌ No video files found in input_videos directory")
    video_path = video_files[0]
    
    print(f"🎬 Loading video: {video_path.name}")
    
    # Setup device
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🖥️ Using device: {device}")
    
    # Load YOLO detector
    model_path = project_root / "models" / "detect" / "best.pt"
    if not model_path.exists():
        raise FileNotFoundError(f"❌ YOLO model not found: {model_path}")
    
    detector = ObjectDetectionProcessor(
        model_path=str(model_path),
        confidence_threshold=0.3
    )
    
    # Load video
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    print(f"📺 Video: {width}x{height} @ {fps} FPS")
    
    # Process frames
    frames = []
    frame_idx = 0
    processed_count = 0
    
    while processed_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
            
        timestamp = frame_idx / fps
        detections = detector._detect_objects(frame)
        
        frame_data = FrameData(
            frame_number=frame_idx,
            timestamp=timestamp,
            raw_frame=frame,
            detections=detections,
            metadata={'source': 'real_video', 'device': device}
        )
        frames.append(frame_data)
        
        if processed_count % 10 == 0:
            print(f"📊 Frame {processed_count}: {len(detections)} detections")
        
        frame_idx += 1
        processed_count += 1
    
    cap.release()
    
    # Create VideoData
    video_data = VideoData(
        video_path=str(video_path),
        frame_rate=fps,
        resolution=(width, height),
        duration=len(frames) / fps,
        frames=frames,
        metadata={'source': 'performance_evaluation'}
    )
    
    total_detections = sum(len(f.detections) for f in frames)
    player_count = sum(1 for f in frames for d in f.detections if d.object_type == ObjectType.PLAYER)
    
    print(f"✅ Loaded {len(frames)} frames with {total_detections} detections")
    print(f"👥 Player detections: {player_count}")
    
    return video_data

# Load the data
try:
    video_data = load_video_data(max_frames=30)
except Exception as e:
    print(f"❌ Failed to load video data: {e}")
    video_data = None

In [ ]:
# 🤖 Initialize SigLIP Processor
def create_siglip_processor():
    """Create SigLIP processor with optimal settings"""
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    batch_size = 16 if device == "cuda" else 8
    
    print(f"🤖 Initializing SigLIP processor on {device}")
    print(f"📦 Batch size: {batch_size}")
    
    try:
        processor = SigLIPTeamAssignmentProcessor(
            device=device,
            batch_size=batch_size,
            n_clusters=2
        )
        print("✅ SigLIP processor initialized successfully!")
        return processor
        
    except Exception as e:
        print(f"❌ SigLIP initialization failed: {e}")
        
        # Try CPU fallback if GPU failed
        if device == "cuda":
            print("🔄 Trying CPU fallback...")
            try:
                processor = SigLIPTeamAssignmentProcessor(
                    device="cpu",
                    batch_size=8,
                    n_clusters=2
                )
                print("✅ SigLIP processor initialized on CPU!")
                return processor
            except Exception as cpu_error:
                print(f"❌ CPU fallback also failed: {cpu_error}")
        
        return None

# Create processor
siglip_processor = create_siglip_processor()

In [ ]:
# 🎯 Run Performance Evaluation
def evaluate_siglip_performance(processor, video_data):
    """Comprehensive performance evaluation of SigLIP processor"""
    
    if not processor or not video_data:
        print("❌ Missing processor or video data")
        return None
    
    # Check player count
    total_players = sum(len([d for d in frame.detections if d.object_type == ObjectType.PLAYER]) 
                       for frame in video_data.frames)
    
    if total_players == 0:
        print("❌ No player detections found")
        return None
    
    print(f"🚀 PERFORMANCE EVALUATION")
    print(f"👥 Total players: {total_players}")
    print(f"🎬 Frames: {len(video_data.frames)}")
    
    # Measure training time
    print("\n🔄 Training SigLIP processor...")
    start_time = time.time()
    processor.train(video_data)
    training_time = time.time() - start_time
    print(f"✅ Training completed: {training_time:.2f}s")
    
    # Measure processing time
    print("🔄 Processing video...")
    start_time = time.time()
    processed_video = processor.process(video_data)
    processing_time = time.time() - start_time
    print(f"✅ Processing completed: {processing_time:.2f}s")
    
    # Analyze results
    team_assignments = {}
    confidence_scores = []
    
    for frame in processed_video.frames:
        for detection in frame.detections:
            if (detection.object_type == ObjectType.PLAYER and 
                detection.metadata and 'team_id' in detection.metadata):
                
                team_id = detection.metadata['team_id']
                team_assignments[team_id] = team_assignments.get(team_id, 0) + 1
                
                confidence = detection.metadata.get('assignment_confidence', detection.confidence)
                confidence_scores.append(confidence)
    
    # Calculate metrics
    fps_estimate = len(video_data.frames) / processing_time if processing_time > 0 else 0
    avg_confidence = np.mean(confidence_scores) if confidence_scores else 0
    
    total_assigned = sum(team_assignments.values())
    team_balance = min(team_assignments.values()) / max(team_assignments.values()) if team_assignments and max(team_assignments.values()) > 0 else 0
    
    results = {
        'training_time': training_time,
        'processing_time': processing_time,
        'fps_estimate': fps_estimate,
        'team_assignments': team_assignments,
        'total_assigned': total_assigned,
        'avg_confidence': avg_confidence,
        'team_balance': team_balance,
        'processed_video': processed_video
    }
    
    print(f"\n📊 RESULTS:")
    print(f"⏱️  Training: {training_time:.2f}s")
    print(f"🚀 Processing: {processing_time:.2f}s")
    print(f"🎬 Estimated FPS: {fps_estimate:.1f}")
    print(f"👥 Assigned players: {total_assigned}")
    print(f"🎯 Avg confidence: {avg_confidence:.3f}")
    print(f"⚖️  Team balance: {team_balance:.3f}")
    
    for team_id, count in team_assignments.items():
        print(f"   Team {team_id}: {count} players")
    
    return results

# Run evaluation
if video_data and siglip_processor:
    evaluation_results = evaluate_siglip_performance(siglip_processor, video_data)
else:
    print("❌ Cannot run evaluation - missing data or processor")
    evaluation_results = None

In [ ]:
# 📊 Performance Visualization
def visualize_performance(results):
    """Create performance visualization dashboard"""
    
    if not results:
        print("❌ No results to visualize")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('🎯 SigLIP Performance Analysis Dashboard', fontsize=16, fontweight='bold')
    
    # 1. Processing performance
    metrics = ['Training', 'Processing']
    times = [results['training_time'], results['processing_time']]
    colors = ['#3498db', '#e74c3c']
    
    bars = axes[0, 0].bar(metrics, times, color=colors, alpha=0.7)
    axes[0, 0].set_title('Processing Times')
    axes[0, 0].set_ylabel('Time (seconds)')
    axes[0, 0].grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar, time_val in zip(bars, times):
        height = bar.get_height()
        axes[0, 0].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                       f'{time_val:.2f}s', ha='center', va='bottom')
    
    # 2. Team distribution
    if results['team_assignments']:
        teams = list(results['team_assignments'].keys())
        counts = list(results['team_assignments'].values())
        colors = ['#e74c3c', '#3498db']
        
        wedges, texts, autotexts = axes[0, 1].pie(counts, labels=[f'Team {t}' for t in teams], 
                                                 colors=colors[:len(teams)], autopct='%1.1f%%',
                                                 startangle=90)
        axes[0, 1].set_title('Team Distribution')
    else:
        axes[0, 1].text(0.5, 0.5, 'No team assignments', ha='center', va='center')
        axes[0, 1].set_title('Team Distribution')
    
    # 3. Performance metrics
    metric_names = ['FPS Estimate', 'Avg Confidence', 'Team Balance']
    metric_values = [results['fps_estimate'], results['avg_confidence'], results['team_balance']]
    colors = ['#2ecc71', '#f39c12', '#9b59b6']
    
    bars = axes[1, 0].bar(metric_names, metric_values, color=colors, alpha=0.7)
    axes[1, 0].set_title('Quality Metrics')
    axes[1, 0].set_ylabel('Score')
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    axes[1, 0].tick_params(axis='x', rotation=45)
    
    # Add value labels
    for bar, value in zip(bars, metric_values):
        height = bar.get_height()
        axes[1, 0].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                       f'{value:.2f}', ha='center', va='bottom')
    
    # 4. Performance summary text
    axes[1, 1].axis('off')
    
    summary_text = f"""
PERFORMANCE SUMMARY

⏱️  Training Time: {results['training_time']:.2f}s
🚀 Processing Time: {results['processing_time']:.2f}s
🎬 Estimated FPS: {results['fps_estimate']:.1f}

👥 Total Assigned: {results['total_assigned']}
🎯 Avg Confidence: {results['avg_confidence']:.3f}
⚖️  Team Balance: {results['team_balance']:.3f}

PERFORMANCE RATING:
{'✅ Excellent' if results['fps_estimate'] > 20 else '⚠️ Good' if results['fps_estimate'] > 10 else '❌ Needs Optimization'}
"""
    
    axes[1, 1].text(0.1, 0.9, summary_text, transform=axes[1, 1].transAxes,
                    fontsize=12, verticalalignment='top', fontfamily='monospace')
    
    plt.tight_layout()
    plt.show()

# Create visualization
if evaluation_results:
    visualize_performance(evaluation_results)
else:
    print("❌ No results to visualize")

In [ ]:
# 💡 Performance Analysis & Recommendations
def analyze_performance(results):
    """Analyze performance and provide optimization recommendations"""
    
    if not results:
        print("❌ No results to analyze")
        return
    
    print("🏆 SIGLIP PERFORMANCE ANALYSIS")
    print("=" * 50)
    
    # Performance assessment
    fps = results['fps_estimate']
    confidence = results['avg_confidence']
    balance = results['team_balance']
    
    print(f"\n⚡ REAL-TIME PERFORMANCE:")
    if fps > 20:
        print(f"   ✅ Excellent - Real-time capable ({fps:.1f} FPS)")
    elif fps > 10:
        print(f"   ⚠️ Good - Near real-time ({fps:.1f} FPS)")
    else:
        print(f"   ❌ Poor - Below real-time ({fps:.1f} FPS)")
    
    print(f"\n🎯 ASSIGNMENT QUALITY:")
    if confidence > 0.8:
        print(f"   ✅ High confidence assignments ({confidence:.3f})")
    elif confidence > 0.6:
        print(f"   ⚠️ Moderate confidence assignments ({confidence:.3f})")
    else:
        print(f"   ❌ Low confidence assignments ({confidence:.3f})")
    
    print(f"\n⚖️  TEAM BALANCE:")
    if balance > 0.8:
        print(f"   ✅ Well balanced teams ({balance:.3f})")
    elif balance > 0.6:
        print(f"   ⚠️ Moderately balanced teams ({balance:.3f})")
    else:
        print(f"   ❌ Imbalanced teams ({balance:.3f})")
    
    # Recommendations
    print(f"\n💡 OPTIMIZATION RECOMMENDATIONS:")
    
    if fps < 15:
        print("   🚀 Performance Optimization:")
        print("     - Use GPU acceleration if available")
        print("     - Increase batch size for GPU processing")
        print("     - Consider model quantization")
    
    if confidence < 0.7:
        print("   🎯 Quality Improvement:")
        print("     - Adjust clustering parameters")
        print("     - Experiment with UMAP settings")
        print("     - Consider feature normalization")
    
    if balance < 0.7:
        print("   ⚖️  Balance Improvement:")
        print("     - Review clustering algorithm settings")
        print("     - Check for detection bias")
        print("     - Consider post-processing smoothing")
    
    print(f"\n🚀 NEXT STEPS:")
    print("   1. Test on diverse video datasets")
    print("   2. Compare with ground truth annotations")
    print("   3. Implement temporal consistency")
    print("   4. Deploy in production pipeline")
    
    print("\n" + "=" * 50)
    print("🎉 Performance evaluation complete!")

# Run analysis
if evaluation_results:
    analyze_performance(evaluation_results)
else:
    print("❌ No results to analyze")